# W09 tutorial — the Item 1A Tone Tracker

**ACFI20009 Financial Data — 26/27**  
Dr Zeming Li, University of Bristol

**This week you build the Item 1A Tone Tracker:** a small pipeline that turns risk-factor sentences into a firm-year tone series, built from two parquet files — the EDGAR sentence snapshot and the Loughran–McDonald dictionary.

Work straight down the notebook. Each section explains what we are doing and why, then does it. By the last cell the artefact exists and runs.

> ### 🤖 Working with Gemini in this notebook
>
> Colab has Gemini built in. You are **encouraged** to use it — reading and
> directing AI-written code is part of what this unit teaches.
>
> **One hard rule:** never paste rows of CRSP data into Gemini (or any other
> AI tool). The licence forbids it. Describe the *schema* instead — column
> names, dtypes, what a row means — which is all an assistant needs to write
> correct code.

## Setup

Run these two cells once, top to bottom, before anything else.

In [ ]:
# Nothing to install: this notebook uses only what Colab already ships
# (pandas, numpy, matplotlib, statsmodels, scipy, sklearn, pyarrow).

In [ ]:
# --- Data -------------------------------------------------------------------
# The unit's datasets are distributed through Blackboard. Download the
# `ACFI20009_data` folder once, put it in your own Google Drive, and this cell
# will find it. Nothing licensed is ever stored in the public repository, and
# the data never leaves your personal Drive.
from pathlib import Path

try:                                    # running on Colab
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/ACFI20009/data")
except ModuleNotFoundError:             # running locally (convenor)
    DATA = Path("2627/data")

assert DATA.exists(), (
    f"Data folder not found at {DATA}.\n"
    "Download it from Blackboard and place it at MyDrive/ACFI20009/data"
)
print("Data:", DATA, "|", len(list(DATA.glob('*.parquet'))), "parquet files")

---

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

In [ ]:
sents = pd.read_parquet(DATA / "edgar_10k_sentences.parquet")
print(sents.shape)
print(sents["company"].unique())
sents[["cik", "company", "filing_year", "item", "sent_id"]].head(3)

In [ ]:
print(sents["text"].iloc[0][:300])

Somebody's lawyer wrote every one of these sentences. Our job is to make them add up to something. Note the `cik` column: the SEC's permanent firm identifier — same stability lesson as permno versus ticker from Week 03.

### 1. The liability demo

We load the Loughran–McDonald dictionary and put the lecture's central claim to the test: generic "negative" words are often ordinary business vocabulary.

In [ ]:
lm = pd.read_parquet(DATA / "lm_dictionary.parquet")
lm["word"] = lm["word"].str.lower()
neg_words = set(lm.loc[lm["sentiment"] == "negative", "word"])
pos_words = set(lm.loc[lm["sentiment"] == "positive", "word"])

for w in ["liability", "tax", "cost", "capital", "loss", "impairment"]:
    print(f"{w:<12} in LM negative list: {w in neg_words}")

`liability`, `tax`, `cost` and `capital` come back `False`; `loss` and `impairment` come back `True`. A dictionary built from psychology reads "tax liability" as doom; a dictionary built from 10-Ks reads it as Tuesday.

In [ ]:
lm["sentiment"].value_counts()

The negative list dwarfs the positive one — financial prose has a rich vocabulary for trouble and a thin one for joy.

### 2. Tokenise and score

Sentences → tokens → flags → firm-level tone. This is the lecture's hardest block; the Notes (§4.3) walk it line by line.

In [ ]:
words = (
    sents.assign(word=sents["text"].str.lower().str.findall(r"[a-z]+"))
         .explode("word")
         .dropna(subset=["word"])
)
words["neg"] = words["word"].isin(neg_words)
words["pos"] = words["word"].isin(pos_words)

tone = (
    words.groupby(["company", "filing_year"], as_index=False)
         .agg(n_words=("word", "size"),
              n_neg=("neg", "sum"),
              n_pos=("pos", "sum"))
)
tone["neg_per_1000"] = 1000 * tone["n_neg"] / tone["n_words"]
latest_year = tone["filing_year"].mode().max()   # the year most firms filed in
latest = tone.loc[tone["filing_year"] == latest_year]
latest.sort_values("neg_per_1000", ascending=False).head(10)

Every firm lands in the tens of negative words per 1,000 — Item 1A is negative *by construction*. (We compare firms in the filing year most of them share, not the largest `filing_year`: Walmart's fiscal year ends in January, so its newest 10-K carries a later label than anyone else's and would sit in a year of its own.) Say it with your tutor: the level is boring; the **cross-section** and the **change** are the signal.

### 3. What we carry forward

`sents`, `lm`, `neg_words`, `pos_words`, `words`, `tone` and `latest_year` all stay in the session — every section from here builds on them. Work in pairs if you can: one types, one predicts each output before it runs (swap halfway). Predicting before running is the cheapest bug-detector known to science.

---

### 4. Counting the testimony

Before scoring anything we need to know how much testimony each firm files. Two counts, side by side: the number of *sentences* per company per filing year, and — from `words` — the three firms with the longest Item 1A in *words* in the most recent filing year. Watch whether "longest" and "most sentences" give the same ranking.

In [ ]:
# sentences per company-year
sent_counts = (
    sents.groupby(["company", "filing_year"])
         .size()
         .unstack("filing_year", fill_value=0)
)
print(sent_counts)

# words per company in the latest filing year
word_counts = words.loc[words["filing_year"] == latest_year, "company"].value_counts()
print(word_counts.head(3))

The rankings usually agree closely but not perfectly — firms differ in sentence length as well as sentence count. The practical point of the word count is the denominator you will need everywhere: any tone comparison across firms must normalise by document length, or the wordiest filer mechanically "wins".

---

### 5. The full tone table

Negativity alone is half a picture, so we extend `tone` with `pos_per_1000` and `net_per_1000 = pos_per_1000 - neg_per_1000`, then read off the most recent filing year: which firm sounds most negative per 1,000 words, which least, and whether net tone is positive for anybody at all. Predict that last one before the cell runs — what you already know points one way; check whether every firm agrees.

In [ ]:
tone["pos_per_1000"] = 1000 * tone["n_pos"] / tone["n_words"]
tone["net_per_1000"] = tone["pos_per_1000"] - tone["neg_per_1000"]
latest = tone.loc[tone["filing_year"] == latest_year]

cols = ["company", "neg_per_1000", "pos_per_1000", "net_per_1000"]
print(latest.sort_values("neg_per_1000", ascending=False)[cols])
print(f"\nFirms with positive net tone: {(latest['net_per_1000'] > 0).sum()}")

Net tone is negative for nine of the ten firms; the exception is Exxon Mobil, whose Item 1A has far fewer LM-negative words than its peers'. The general pattern was predictable from two facts you already had: Item 1A is a list of hazards (negative by construction), and the LM negative vocabulary is several times larger than the positive one. Neither fact is news about any individual firm — which is precisely why the *level* of tone is a weak signal and the cross-section and change carry the information.

---

### 6. Who hedges most?

The LM dictionary's **uncertainty** list ("may", "approximately", "could", …) measures hedging rather than negativity. We compute `unc_per_1000` per firm for the most recent filing year and set it beside the negativity ranking from section 5. Does the most-hedged firm coincide with the most-negative one — and what does it mean for a firm to rank high on uncertainty but middling on negativity?

In [ ]:
unc_words = set(lm.loc[lm["sentiment"] == "uncertainty", "word"])
words["unc"] = words["word"].isin(unc_words)

hedging = (
    words.groupby(["company", "filing_year"], as_index=False)
         .agg(n_words=("word", "size"), n_unc=("unc", "sum"))
)
hedging["unc_per_1000"] = 1000 * hedging["n_unc"] / hedging["n_words"]
latest_h = hedging.loc[hedging["filing_year"] == latest_year]
print(latest_h.sort_values("unc_per_1000", ascending=False)
              [["company", "unc_per_1000"]])

The rankings are related but rarely identical. High-uncertainty-but-middling-negativity language describes a firm that is not reporting bad outcomes but is unwilling to commit to good ones — lots of "may", "could" and "approximately" around otherwise neutral statements. Tone is not one number; that is why LM ships four lists, and why a serious tone pipeline reports negativity and uncertainty as separate columns rather than collapsing them into one.

---

### 7. Themes have start dates

Dictionaries measure *how* firms talk; substring search measures *what* they talk about. With `str.contains` (and `case=False`, always) we build a theme × filing-year table of hits per 1,000 sentences for three patterns: `cyber`, `pandemic|epidemic|covid`, and `artificial intelligence|machine learning`. Two things to watch in the table: whether pandemic language is still present in filings written after the pandemic, and whether AI language is rising across these firms. The last column holds Walmart's newest filing alone (its fiscal year ends in January), so read it as one firm, not a trend. The harder question comes after it — why a theme might appear in filings *before* it ever moves prices, and why it might appear *after*.

In [ ]:
themes = {
    "cyber":    r"cyber",
    "pandemic": r"pandemic|epidemic|covid",
    "AI":       r"artificial intelligence|machine learning",
}
by_theme = {}
total = sents.groupby("filing_year").size()
for name, pattern in themes.items():
    hits = sents["text"].str.contains(pattern, case=False, regex=True)
    share = (1000 * sents.loc[hits].groupby("filing_year").size() / total).round(1)
    by_theme[name] = share
pd.DataFrame(by_theme).fillna(0).T

Pandemic language never goes away: it runs at roughly 5 to 11 hits per 1,000 sentences in every year of the snapshot, long after the pandemic itself (the snapshot starts too late to show when it first appears); cyber is present throughout and climbs from 2023 to 2025; AI mentions are rare, a handful per 1,000 sentences, and edge up only in the 2025 filings. The 2026 column is Walmart alone and is not a trend. On the timing question: a theme appears *before* prices because lawyers must flag a risk as soon as it is material even if it never crystallises (disclosure is cheap insurance); *after* prices, because boilerplate also persists — firms keep listing risks long after the market has priced or dismissed them. Both directions are reasons a raw theme count is not yet a trading signal.

---

### 8. Your first LLM extraction (cached)

Below are ten risk-factor sentences of the kind in the snapshot, and the ten cached replies an LLM produced for them under the lecture's schema (`risk_type` ∈ operational/financial/legal/macro/technology/other; `sentiment` ∈ positive/negative/neutral; `mentions_quantity` ∈ true/false). The convenor ran the model once in July 2026 so no key is needed.

A model's reply is not data until something has checked it, so the first thing we write is `validate_extraction(raw)`: it returns a list of problems, and an empty list means the row is admissible. It tests four things — valid JSON, exactly the three schema keys, both categorical values inside their allowed sets, and `mentions_quantity` as a genuine boolean. We then run it over all ten cached replies, count how many pass, and assemble the admissible rows into a DataFrame (one row per sentence, with `sent_id`) with `risk_type` tabulated.

With an API key you can regenerate the replies live using the lecture's `anthropic` pattern; whatever comes back must clear the same validator.

In [ ]:
demo = pd.DataFrame({
    "sent_id": range(1, 11),
    "text": [
        "A significant portion of our revenue is derived from a small number of large customers, and the loss of any of them could materially reduce our operating results.",
        "Our outstanding indebtedness of approximately $12.5 billion contains covenants that restrict our ability to incur additional debt.",
        "We are subject to ongoing antitrust investigations in the United States and the European Union, which could result in substantial fines.",
        "Fluctuations in foreign exchange rates, particularly the euro and the yen, may adversely affect our reported revenue.",
        "A successful cyberattack on our systems could compromise customer data and expose us to liability and reputational harm.",
        "A prolonged economic downturn in our key markets could reduce demand for our products and services.",
        "We do not believe that the outcome of these legal proceedings will have a material adverse effect on our financial condition or results of operations.",
        "Our manufacturing operations depend on a single facility in Austin, Texas, and any disruption at that site could delay shipments by up to six months.",
        "We face intense competition from established firms and new entrants, which may pressure our pricing and margins.",
        "Failure to attract and retain qualified engineering personnel could impair our ability to develop new products.",
    ],
})

cached_replies = [
    '{"risk_type": "operational", "sentiment": "negative", "mentions_quantity": false}',
    '{"risk_type": "financial", "sentiment": "negative", "mentions_quantity": true}',
    '{"risk_type": "legal", "sentiment": "negative", "mentions_quantity": false}',
    '{"risk_type": "macro", "sentiment": "negative", "mentions_quantity": false}',
    '{"risk_type": "technology", "sentiment": "negative", "mentions_quantity": false}',
    '{"risk_type": "macro", "sentiment": "negative", "mentions_quantity": false}',
    '{"risk_type": "legal", "sentiment": "negative", "mentions_quantity": false}',
    '{"risk_type": "operational", "sentiment": "negative", "mentions_quantity": true}',
    '{"risk_type": "other", "sentiment": "negative", "mentions_quantity": false}',
    '{"risk_type": "operational", "sentiment": "negative", "mentions_quantity": false}',
]

In [ ]:
import json

ALLOWED_RISK = {"operational", "financial", "legal",
                "macro", "technology", "other"}
ALLOWED_SENT = {"positive", "negative", "neutral"}
REQUIRED_KEYS = {"risk_type", "sentiment", "mentions_quantity"}

def validate_extraction(raw: str) -> list[str]:
    """Return a list of problems; an empty list means the row is admissible."""
    problems = []
    try:
        d = json.loads(raw)
    except json.JSONDecodeError:
        return ["not valid JSON"]
    if set(d) != REQUIRED_KEYS:
        problems.append(f"keys are {sorted(d)}, expected {sorted(REQUIRED_KEYS)}")
        return problems
    if d["risk_type"] not in ALLOWED_RISK:
        problems.append(f"risk_type {d['risk_type']!r} not in schema")
    if d["sentiment"] not in ALLOWED_SENT:
        problems.append(f"sentiment {d['sentiment']!r} not in schema")
    if not isinstance(d["mentions_quantity"], bool):
        problems.append("mentions_quantity is not a boolean")
    return problems

# validate all ten
report = [validate_extraction(r) for r in cached_replies]
n_ok = sum(len(p) == 0 for p in report)
print(f"admissible replies: {n_ok} / {len(cached_replies)}")

# DataFrame of the admissible rows, one row per sentence
results = (
    pd.DataFrame([json.loads(r) for r, p in zip(cached_replies, report) if not p])
      .assign(sent_id=[i for i, p in zip(demo["sent_id"], report) if not p])
)
print(results[["sent_id", "risk_type", "sentiment", "mentions_quantity"]])
print(results["risk_type"].value_counts())

All **10 of 10** replies pass the schema check, and the results table shows a sensible spread of risk types (operational 3, legal 2, macro 2, financial 1, technology 1, other 1), with `mentions_quantity` true for exactly the two sentences quoting figures ("\$12.5 billion", "six months"). Schema-perfect — hold that thought for section 9.

---

### 9. Verify the AI

An AI assistant, shown the section 8 batch and the validator's clean report, concluded:

> "All ten outputs conform to the schema: every key is present, every value is in its allowed vocabulary, and all types are correct. **The extraction is therefore correct and the labels are ready to use downstream.**"

Three things are worth settling for yourself before reading on, because the walkthrough below gives them all away. Which step in the assistant's reasoning is valid, and where exactly does the logic break? Reading the ten sentences against their labels, **one label is wrong** — which one, what should it say, and what fooled the model? And since the validator scored 10/10 and still missed it, what would it concretely take to catch this class of error at the scale of 3,000 firms?

**Where the logic breaks.** The premise is true: all ten outputs *are* schema-valid — section 8 established exactly that. The break is the word "**therefore**". Schema checks establish that output is **well-formed**; they say nothing about whether it is **true**. The assistant has quietly upgraded "malformed = 0" to "mistaken = 0" — the exact confusion the lecture warned about: *asserts catch malformed, not mistaken.*

**The wrong label.** The planted flaw is **sentence 7**:

> "We **do not believe** that the outcome of these legal proceedings **will have a material adverse effect** on our financial condition or results of operations."

labelled `"sentiment": "negative"`. The sentence is management explicitly *reassuring* the reader — the correct label under our schema is `"neutral"` (arguably even mildly positive; certainly not negative). What fooled the model is the same trap that fools the dictionary: the sentence is saturated with alarming vocabulary ("legal proceedings", "material adverse effect"), and the single negation "do not believe" flips the meaning of all of it. The `risk_type: "legal"` half of the label is fine — the sentence *is about* legal risk — which makes the wrong sentiment all the more plausible-looking.

The irony is worth savouring: negation is the canonical case where LLMs are supposed to beat word-counting, and here one failed in exactly the dictionary's way, while producing perfectly formatted JSON. Fluency is not accuracy.

**What it would take at scale.** Ground truth: a random sample of sentences **labelled by humans** (with an agreement check between labellers, since humans disagree too), against which you measure the model's precision and recall per class — then a decision rule for whether the error rate is acceptable for the use case. That is precisely the Week 10 agenda, and next week you will build it: hand-label 30 sentences, compute Cohen's kappa against a partner, and score the LLM against your own labels.

If you spotted the flaw before reading this section, write down the reasoning that got you there while it is still fresh. A documented case of catching a plausible-but-wrong AI label — the sentence, the label, why it is wrong — is worth far more than a general suspicion that models make mistakes.